# Week 1-3

In the first 6 Chapters I learned:

- Variables and data types
- Basic operations
- Input and output
- Type conversion
- Lists and tuples
- Dictionaries and sets
- Functions
- Control flow (if statements, loops)
- Basic File Operations

Using this knowledge, I want to build the first part of my Python Portfolio.
The Portfolio itself should eventually become a flexible tool covering the whole machine learning workflow. The first part consists of:

- Sourcing: Tha application should read a text file containing a list of urls, the application should iterate through the list and download the files from the internet or a given path, into the sourcing folder. At the end of the preocess, the application should print out a summary of every step. (url / path, timestamp,  filename, size, time taken to download). For the failed downloads, the user can choose wehther to create a separate list of the config-file for another attempt. It should ask whether we want to retry the failed downloads before next initiation with the original list starts, should the user chooses no.

- Data cleansing: The application should read a text file containing lists of file selection (file name filter), the cleansing function(s), it should apply to the files. The cleaned version should be saved in a separate directory. A summary should be printed out for every operation done for each file.

- Analyze: The user can choose from a list of files (clean data) to display different basic statistics of the data. And the end of the operation, the user can choose whether to choose another file to look at or quit.

From ChatGPT:

🔹 1. Basic Structure and Metadata

- Shape: Number of rows and columns (df.shape)
- Column names and types (df.dtypes)
- Index: Is it meaningful or just default?
- Sample data: View the first few rows (df.head())

🔹 2. Variable (Feature) Information

For each column:
- Name
- Type (Numerical, Categorical, Text, Date/Time, Boolean)
- Unique values (df['col'].unique())
- Missing values (df.isnull().sum())

🔹 3. Descriptive Statistics

Numerical features:
- Mean, median, min, max, std dev (df.describe())
- Distributions (histograms, box plots)
- Categorical features:
- Frequency counts (df['col'].value_counts())
- Mode, cardinality (number of unique values)

🔹 4. Data Quality Checks

- Missing data: Amount, location, and patterns
- Duplicates: Rows or entries (df.duplicated())
- Outliers: Use IQR, Z-score, or visual tools (box plots)
- Inconsistent data: Typos, mixed types in one column

🔹 5. Relationships and Correlations

- Correlation matrix for numerical variables
- Cross-tabulations or groupings for categorical vs numerical
- Pairplots or scatter matrix for visual inspection

🔹 6. Target Variable Understanding (if doing supervised learning)

- Type (Regression or Classification)
- Distribution and balance of classes or range of values
- Any leakage or target-related data issues

🔹 7. Contextual Understanding

- Source of the data
- What each column means (check data dictionary or metadata)
- Domain relevance (is something unusual, or expected?)
- Timeframes involved (especially for time series data)

Optional but Helpful

- Units: Are amounts in USD? Lengths in meters?
- Encoding: Are categories encoded numerically?
- Timezones: For datetime fields


## Initiation

Initiate global variables + Imports

In [ ]:
import urllib.request
import shutil
import os

# Get the current working directory
current_directory = os.getcwd()
print("The current working directory is:", current_directory)

sourceCFG = current_directory+"\\2_Sourcing\\source.cfg"
sourceCFGList = []

#Missing items
MissCFG = current_directory+"\\2_Sourcing\\bkp.cfg"
MissCFGList = []



The current working directory is: h:\Documents\My Training\CAS\CAS_DLS
You chose Sourcing
Starting data sourcing...
Configuration loaded:
https://dam-api.bfs.admin.ch/hub/api/dam/assets/36062343/master,su-q-01.04.00.12.xlsx,\2_Sourcing\su-q-01.04.00.12.xlsx
https://dam-api.bfs.admin.ch/hub/api/dam/assets/36163086/master,ts-x-16.02.01-01.csv,\2_Sourcing\ts-x-16.02.01-01.csv
https://dam-api.bfs.admin.ch/hub/api/dam/assets/36162553/master,ts-q-01.04.02.01.30-P.csv,\2_Sourcing\ts-q-01.04.02.01.30-P.csv
\2_Sourcing\ts-x-16.02.01-01.csv
Exiting the program.


## 2. Read Config Function

Should read each line from a cfg file (text) and store it in a list

In [ ]:
def read_config(file_path):
    try:
        with open(file_path, 'r') as file:
            config = file.read()
            for line in config.splitlines():
                #print(line.strip())
                sourceCFGList.append(line.strip())                
            print("Configuration loaded:")
            print(config)
    except FileNotFoundError:
        print(f"Configuration file '{file_path}' not found.")
        return None

## 4. Write Config

Write back the missing files into a configuration file for the next run.

- Parameters: config file path, ArrayList (url, filename, destination)

In [ ]:
def write_config(file_path, data):
    try:
        with open(file_path, 'w') as file:
            for line in data:
                file.write(','.join(line) + '\n')
        print(f"Configuration saved to '{file_path}'")
    except Exception as e:
        print(f"Error writing configuration file: {e}")
        return None

## Download File
Retrieve a file from url and save it to the destination path

In [ ]:

def download_file(url, destination):
    try:
        urllib.request.urlretrieve(url, destination)
        print(f"File downloaded successfully: {destination}")
    except Exception as e:
        print(f"Error downloading file: {e}")
        return None


## Move File

Move file from one folder to another

In [ ]:
def move_file(source, destination):
    try:
        os.rename("path/to/current/file.foo", "path/to/new/destination/for/file.foo")
        os.replace("path/to/current/file.foo", "path/to/new/destination/for/file.foo")
        shutil.move("path/to/current/file.foo", "path/to/new/destination/for/file.foo")
        shutil.move(source, destination)
        print(f"File moved successfully from {source} to {destination}")
    except Exception as e:
        print(f"Error moving file: {e}")

## Source List
Iterate through a list and automatically chooses to download or move file according to the url, and save the file in a the destination folder.

- Parameter list of Array ( URL, Filename, Destination)

In [ ]:
def source_list(CFGList):
    NewMissCFGList = []
    try:
        for sourceCFG in CFGList:
            url = sourceCFG[0]
            filename = sourceCFG[1]
            destination = sourceCFG[2]
            if url.index("http") != -1:
                try:
                    move_file(url, destination)
                except Exception as e:
                    print(f"Error moving file: {e}")
                    NewMissCFGList.append([url,filename,destination])
            else:
                try:
                    download_file(url, filename, destination)
                except Exception as e:
                    print(f"Error downloading file: {e}")
                    NewMissCFGList.append([url,filename,destination])
        write_config(MissCFG, NewMissCFGList)
    except Exception as e:
        print(f"Critical error sourcing file: {e}") 
        return None

## Main Loop

Iterate until the user is satisfied

In [ ]:

if __name__ == "__main__":
    exit = False
    while exit != True:
        response = input("Please choose processing mode: \n 1. Sourcing \n 2. Data Cleansing\n 3. Analyze\n 4. Results\n 5. Walkthrough\n 6. Exit")
        if response == '1':
            print("You chose Sourcing")
            #check if file exists (bkp.cfg)
            if os.path.exists(MissCFG):
                response = input("Would you like to try sourcing the missing data? (y/n): ").strip().lower()
                if response == 'y': 
                    read_config(MissCFG)
                    #check whether list exists
                    if MissCFGList:
                        print(MissCFGList[0])
                        source_list(MissCFGList)

            if os.path.exists(sourceCFG):    
                response = input("Would you like to start sourcing the data? (y/n): ").strip().lower()
                if response == 'y':
                    print("Starting data sourcing...")
                    read_config(sourceCFG)
                    # Example: Download a file from a URL specified in the config from WEB
                    #urllib.request.urlretrieve("http://www.example.com/songs/mp3.mp3", "mp3.mp3")
                    if sourceCFGList:
                        print(sourceCFGList[1].split(',')[2])
            else:
                print(f"Source configuration file '{sourceCFG}' not found.")

        elif response == '6':
            exit = True
            print("Exiting the program.")